In [1]:
import subprocess
subprocess.run(["pip", "install", "timm>=0.9.12", "-q"], check=True)
print("Dependencies ready.")

Dependencies ready.


In [2]:
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import timm

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm import tqdm

print("Imports OK")
print("PyTorch :", torch.__version__)
print("timm    :", timm.__version__)

Imports OK
PyTorch : 2.10.0+cu128
timm    : 1.0.25


In [3]:
# Enforce GPU — if this fails, go to Session options → Accelerator → GPU T4 x1
if not torch.cuda.is_available():
    raise EnvironmentError(
        "GPU not available. "
        "Enable it: Session options → Accelerator → GPU T4 x1"
    )

DEVICE = torch.device("cuda")

print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM total   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free    : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

GPU          : Tesla T4
VRAM total   : 15.6 GB
VRAM free    : 15.5 GB


In [4]:
# ── path setup ──────────────────────────────────────────────────────
DATASET_ROOT = Path("/kaggle/input/datasets/alistairking/recyclable-and-household-waste-classification")
SPLITS_CSV   = Path("/kaggle/input/datasets/nethmihirunika/splits-csv/splits.csv")
CONFIG_PATH  = Path("/kaggle/input/datasets/nethmihirunika/config-yaml/config.yaml")
OUTPUT_DIR   = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths exist
for name, path in [("Dataset", DATASET_ROOT),
                    ("splits.csv", SPLITS_CSV),
                    ("config.yaml", CONFIG_PATH)]:
    status = "OK" if path.exists() else "NOT FOUND — check Add Input"
    print(f"  {name:<15}: {path}  [{status}]")

  Dataset        : /kaggle/input/datasets/alistairking/recyclable-and-household-waste-classification  [OK]
  splits.csv     : /kaggle/input/datasets/nethmihirunika/splits-csv/splits.csv  [OK]
  config.yaml    : /kaggle/input/datasets/nethmihirunika/config-yaml/config.yaml  [OK]


In [5]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SEED         = cfg["data"]["seed"]                          # 42
IMG_SIZE     = cfg["data"]["image_size"]                    # 224
NUM_CLASSES  = cfg["data"]["num_classes"]                   # 6
BATCH_SIZE   = cfg["training"]["batch_size"]                # 32
NUM_EPOCHS   = cfg["training"]["num_epochs"]                # 20
PATIENCE     = cfg["training"]["early_stopping_patience"]  # 5
LR           = cfg["optimizer"]["lr"]                       # 0.0001
WEIGHT_DECAY = cfg["optimizer"]["weight_decay"]             # 0.01
MIN_LR       = cfg["scheduler"]["min_lr"]                   # 0.000001
CLASS_MAP    = cfg["classes"]                               # {Plastic:0,...}


NUM_WORKERS  = 2


CKPT_PATH    = OUTPUT_DIR / "best_model.pth"

# Reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
# benchmark=True speeds up training when input size is fixed (always 224x224)
torch.backends.cudnn.benchmark = True

print("Config loaded.")
print(f"  Seed        : {SEED}")
print(f"  Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  LR          : {LR}")
print(f"  Checkpoint  : {CKPT_PATH}")

Config loaded.
  Seed        : 42
  Image size  : 224x224
  Batch size  : 32
  Epochs      : 20
  LR          : 0.0001
  Checkpoint  : /kaggle/working/best_model.pth


In [6]:
df = pd.read_csv(SPLITS_CSV)


def remap_path(local_path: str) -> str:
    
    # Normalise separators
    p = local_path.replace("\\", "/")
    # Find 'images/' in path and keep everything after it
    marker = "images/"
    idx = p.find(marker)
    if idx == -1:
        raise ValueError(f"Cannot find '{marker}' in path: {p}")
    relative = p[idx + len(marker):]   # e.g. 'Plastic water bottles/default/img1.png'
    return str(DATASET_ROOT / "images" / "images" / relative)

df["filepath"] = df["filepath"].apply(remap_path)

# Verify a few paths exist
sample_paths = df["filepath"].sample(3, random_state=SEED).tolist()
print("Sample path check:")
for p in sample_paths:
    print(f"  {'OK' if Path(p).exists() else 'MISSING'} — {p}")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print(f"\nTrain : {len(train_df)} images")
print(f"Val   : {len(val_df)} images")
print(f"Test  : {len(test_df)} images")
print(f"\nClass distribution in train:")
print(train_df["main_class"].value_counts().to_string())

Sample path check:
  OK — /kaggle/input/datasets/alistairking/recyclable-and-household-waste-classification/images/images/plastic_food_containers/real_world/Image_156.png
  OK — /kaggle/input/datasets/alistairking/recyclable-and-household-waste-classification/images/images/plastic_cup_lids/default/Image_139.png
  OK — /kaggle/input/datasets/alistairking/recyclable-and-household-waste-classification/images/images/shoes/real_world/Image_54.png

Train : 10500 images
Val   : 2250 images
Test  : 2250 images

Class distribution in train:
main_class
Plastic    4550
Paper      2100
Metal      1400
Organic    1050
Textile     700
Glass       700


In [7]:
class TrashDataset(Dataset):
    """PyTorch Dataset reading filepaths from splits.csv.
    Transforms passed in — training uses augmentation,
    val/test use only resize + normalize.
    """
    def __init__(self, df: pd.DataFrame, transform=None):
        self.filepaths = df["filepath"].values
        self.labels    = df["class_idx"].values
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])

print("TrashDataset defined.")

TrashDataset defined.


In [8]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training — augmentation only here, never in val/test
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val/test — resize + normalize only
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms defined.")
print("  Train : resize + augment + normalize")
print("  Val   : resize + normalize only")

Transforms defined.
  Train : resize + augment + normalize
  Val   : resize + normalize only


In [9]:
train_dataset = TrashDataset(train_df, transform=train_transform)
val_dataset   = TrashDataset(val_df,   transform=val_transform)
test_dataset  = TrashDataset(test_df,  transform=val_transform)

# WeightedRandomSampler — oversample minority classes (Glass, Textile)
train_labels   = train_df["class_idx"].values
class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[label] for label in train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

train_loader = DataLoader(train_dataset, sampler=sampler, **loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False,   **loader_kwargs)
test_loader  = DataLoader(test_dataset,  shuffle=False,   **loader_kwargs)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 329
Val batches   : 71
Test batches  : 71


In [10]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
class_weights = torch.FloatTensor(weights).to(DEVICE)

print("Class weights (higher = minority class penalised more):")
for cls_name, idx in CLASS_MAP.items():
    print(f"  {cls_name:<10}  idx={idx}  weight={weights[idx]:.4f}")

Class weights (higher = minority class penalised more):
  Plastic     idx=0  weight=0.3846
  Paper       idx=1  weight=0.8333
  Glass       idx=2  weight=2.5000
  Metal       idx=3  weight=1.2500
  Organic     idx=4  weight=1.6667
  Textile     idx=5  weight=2.5000


In [11]:
backbone = timm.create_model(
    "mobilenetv3_large_100",
    pretrained=True,
    num_classes=0,
    global_pool="avg"
)

# Freeze all backbone layers for Stage 1
for param in backbone.parameters():
    param.requires_grad = False

# Get feature dimension
with torch.no_grad():
    feat_dim = backbone(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
print(f"Backbone feature dim: {feat_dim}")

class MobileNetTrash(nn.Module):
    def __init__(self, backbone, feat_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = MobileNetTrash(backbone, feat_dim, NUM_CLASSES).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,} total")
print("Backbone frozen for Stage 1.")
print(f"VRAM after model init : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Backbone feature dim: 1280
Trainable params : 332,038 / 4,534,070 total
Backbone frozen for Stage 1.
VRAM after model init : 0.02 GB


In [12]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    """GPU + AMP training epoch."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        # non_blocking speeds up CPU→GPU transfer when pin_memory=True
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        # Forward in float16
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)

        # Scaled backward — prevents float16 underflow
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """GPU + AMP evaluation."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="  Eval ", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def save_checkpoint(model, path, epoch, val_acc):
    """Save full checkpoint dict to /kaggle/working/."""
    torch.save({
        "epoch"       : epoch,
        "val_acc"     : val_acc,
        "model_state" : model.state_dict(),
        "model_name"  : "mobilenetv3_large_100",
        "num_classes" : NUM_CLASSES,
    }, path)
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"  Saved → {path}  val_acc={val_acc:.4f}  VRAM={vram:.2f}GB")


print("Helpers defined.")

Helpers defined.


In [13]:
# Stage 1: Train head only — backbone frozen
EPOCHS_STAGE1 = 10
LR_STAGE1     = 1e-3

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE1, weight_decay=WEIGHT_DECAY
)
scheduler1 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer1, T_max=EPOCHS_STAGE1, eta_min=MIN_LR
)
# AMP scaler — add this before Stage 1 training loop
scaler = torch.cuda.amp.GradScaler()

best_val_acc   = 0.0
patience_count = 0
history        = {"train_loss": [], "train_acc": [],
                  "val_loss":   [], "val_acc":   []}

print("=" * 58)
print("  Stage 1 — Head only  (GPU + AMP + WeightedSampler)")
print("=" * 58)

for epoch in range(1, EPOCHS_STAGE1 + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer1, criterion, scaler
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler1.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    vram = torch.cuda.memory_allocated() / 1e9
    print(f"  Ep {epoch:>2}/{EPOCHS_STAGE1}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
          f"VRAM={vram:.2f}GB")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, CKPT_PATH, epoch, val_acc)
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}.")
            break

print(f"\nStage 1 complete. Best val_acc = {best_val_acc:.4f}")

/tmp/ipykernel_101/3412515245.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  Stage 1 — Head only  (GPU + AMP + WeightedSampler)


  Train:   0%|          | 0/329 [00:00<?, ?it/s]/tmp/ipykernel_101/1284697548.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval :   0%|          | 0/71 [00:00<?, ?it/s]           /tmp/ipykernel_101/1284697548.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Ep  1/10  train_loss=0.4578  train_acc=0.7516  val_loss=0.5000  val_acc=0.7151  VRAM=0.04GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.7151  VRAM=0.04GB


  Ep  2/10  train_loss=0.3364  train_acc=0.8179  val_loss=0.5060  val_acc=0.7293  VRAM=0.04GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.7293  VRAM=0.04GB


...

  Ep  6/10  train_loss=0.1974  train_acc=0.8793  val_loss=0.3538  val_acc=0.8080  VRAM=0.04GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.8080  VRAM=0.04GB


  Ep  7/10  train_loss=0.1902  train_acc=0.8818  val_loss=0.3446  val_acc=0.8338  VRAM=0.04GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.8338  VRAM=0.04GB


  Ep  8/10  train_loss=0.1761  train_acc=0.8880  val_loss=0.3577  val_acc=0.8200  VRAM=0.04GB


  Ep  9/10  train_loss=0.1700  train_acc=0.8872  val_loss=0.3581  val_acc=0.8200  VRAM=0.04GB


  Ep 10/10  train_loss=0.1703  train_acc=0.8926  val_loss=0.3539  val_acc=0.8253  VRAM=0.04GB

Stage 1 complete. Best val_acc = 0.8338


In [14]:
# Stage 2: Unfreeze top MobileNet blocks and fine-tune
EPOCHS_STAGE2 = 10
LR_STAGE2     = 1e-5

for param in model.backbone.parameters():
    param.requires_grad = True

# Unfreeze only last 3 blocks — keep early layers frozen
UNFREEZE_BLOCKS = ["blocks.4", "blocks.5", "blocks.6", "conv_head"]
for name, param in model.backbone.named_parameters():
    if not any(b in name for b in UNFREEZE_BLOCKS):
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after unfreeze: {trainable:,}")

optimizer2 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE2, weight_decay=WEIGHT_DECAY
)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=EPOCHS_STAGE2, eta_min=MIN_LR
)

patience_count = 0

print("=" * 58)
print("  Stage 2 — Fine-tuning top blocks  (GPU + AMP)")
print("=" * 58)

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer2, criterion, scaler
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler2.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    vram = torch.cuda.memory_allocated() / 1e9
    print(f"  Ep {epoch:>2}/{EPOCHS_STAGE2}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
          f"VRAM={vram:.2f}GB")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, CKPT_PATH, epoch, val_acc)
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}.")
            break

print(f"\nStage 2 complete. Best val_acc = {best_val_acc:.4f}")

Trainable params after unfreeze: 4,342,126
  Stage 2 — Fine-tuning top blocks  (GPU + AMP)


  Train:   0%|          | 0/329 [00:00<?, ?it/s]/tmp/ipykernel_101/1284697548.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Eval :   0%|          | 0/71 [00:00<?, ?it/s]           /tmp/ipykernel_101/1284697548.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Ep  1/10  train_loss=0.1546  train_acc=0.8980  val_loss=0.3606  val_acc=0.8129  VRAM=0.09GB


  Ep  2/10  train_loss=0.1537  train_acc=0.9001  val_loss=0.3437  val_acc=0.8289  VRAM=0.09GB


  Ep  3/10  train_loss=0.1398  train_acc=0.9091  val_loss=0.3405  val_acc=0.8236  VRAM=0.09GB


...

  Ep  6/10  train_loss=0.1212  train_acc=0.9156  val_loss=0.3114  val_acc=0.8409  VRAM=0.09GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.8409  VRAM=0.09GB


  Ep  7/10  train_loss=0.1167  train_acc=0.9176  val_loss=0.3297  val_acc=0.8209  VRAM=0.09GB


  Ep  8/10  train_loss=0.1188  train_acc=0.9199  val_loss=0.3161  val_acc=0.8418  VRAM=0.09GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.8418  VRAM=0.09GB


  Ep  9/10  train_loss=0.1120  train_acc=0.9245  val_loss=0.3076  val_acc=0.8444  VRAM=0.09GB
  Saved → /kaggle/working/best_model.pth  val_acc=0.8444  VRAM=0.09GB


  Ep 10/10  train_loss=0.1143  train_acc=0.9190  val_loss=0.3144  val_acc=0.8431  VRAM=0.09GB

Stage 2 complete. Best val_acc = 0.8444


In [15]:
total_epochs = len(history["train_loss"])
epochs_range = range(1, total_epochs + 1)
stage2_start = EPOCHS_STAGE1 + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], label="Train", color="#378ADD")
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   color="#D85A30")
axes[0].axvline(stage2_start, color="gray", linestyle="--", linewidth=1,
                label="Stage 2 start")
axes[0].set_title("Loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
sns.despine(ax=axes[0])

axes[1].plot(epochs_range, history["train_acc"], label="Train", color="#378ADD")
axes[1].plot(epochs_range, history["val_acc"],   label="Val",   color="#D85A30")
axes[1].axvline(stage2_start, color="gray", linestyle="--", linewidth=1,
                label="Stage 2 start")
axes[1].set_title("Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()
sns.despine(ax=axes[1])

fig.suptitle("MobileNetV3-Large — Training History", fontsize=13, fontweight="bold")
fig.tight_layout()
plot_path = OUTPUT_DIR / "training_history.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {plot_path}")

Saved → /kaggle/working/training_history.png


In [16]:
# Load best checkpoint
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
print(f"Loaded checkpoint  epoch={checkpoint['epoch']}  "
      f"val_acc={checkpoint['val_acc']:.4f}")

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

Loaded checkpoint  epoch=9  val_acc=0.8444


  Eval :   0%|          | 0/71 [00:00<?, ?it/s]/tmp/ipykernel_101/1284697548.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                        


Test Loss     : 0.2802
Test Accuracy : 0.8578


In [17]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Collecting preds"):
        images = images.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast():
            outputs = model(images)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
class_names = list(CLASS_MAP.keys())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

  with torch.cuda.amp.autocast():

Classification Report:
              precision    recall  f1-score   support

     Plastic       0.97      0.75      0.84       975
       Paper       0.85      0.91      0.88       450
       Glass       0.65      0.94      0.77       150
       Metal       0.79      0.96      0.86       300
     Organic       0.83      0.97      0.89       225
     Textile       0.87      0.96      0.91       150

    accuracy                           0.86      2250
   macro avg       0.83      0.91      0.86      2250
weighted avg       0.88      0.86      0.86      2250



In [18]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True", fontsize=11)
ax.set_title("MobileNetV3-Large — Confusion Matrix (Test Set)",
             fontsize=12, fontweight="bold", pad=10)
fig.tight_layout()

cm_path = OUTPUT_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {cm_path}")
print(f"\nAll outputs in: {OUTPUT_DIR}")
print("Download best_model.pth from the Output tab and share via Google Drive.")

Saved → /kaggle/working/confusion_matrix.png

All outputs in: /kaggle/working
Download best_model.pth from the Output tab and share via Google Drive.
